### Imports

In [3]:
from ray_tracing_simulator_nnModules_grad import PrismMirror, Ray, Plane, ReflectingPlane, RefractingPlane, Camera, visualize_camera_configuration, closest_point, rotx, get_rot_mat
import matplotlib.pyplot as plt
import numpy as np  
import torch
import random
seed = 0
# Python random
random.seed(seed)
# NumPy random
np.random.seed(seed)
# PyTorch random
torch.manual_seed(seed)
import scipy.io as sio
import os
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, random_split, Dataset
torch.autograd.set_detect_anomaly(True)
import datetime
import time
from arenas.prism_arenas import Arena_reprojection_loss_two_cameras_prism_grid_distances
from utils import euclidean_distance #NOTE to self: Maybe this is not needed. Just use the in-built function?
from torch.utils.tensorboard import SummaryWriter
import argparse
import yaml

### Define the dataloader class

In [4]:
class CalibrationDataset(Dataset):
    def __init__(self, data, labels_2D, labels_3D, pairwise_distance):
        # Example data
        self.data = data.T
        self.labels_2D = labels_2D.T # These are the 2D coordinates of the grid points in the image frame
        self.labels_3D = labels_3D.T # These are known 3D coordinates of the grid points in the world frame (These will not be used in training, only for evaluation)
        self.pairwise_distance = pairwise_distance # These are the known pairwise distances between the grid points based on the grid's geometry (These will be used in training)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels_2D[idx], self.labels_3D[idx], self.pairwise_distance[idx]

log_file = ''

### User inputs

In [33]:
calibration_scripts_logs = 'calibration_scripts_logs'
os.makedirs(calibration_scripts_logs, exist_ok=True)
exp_id = 'exp_32'
num_epochs = 300
log_file = 'exp_32.log'
load_checkpoint = False 
calibration_results_dir = f'./calibration_initialization_data_for_testing'
datatype = torch.float64
input_model_checkpoint_dir = ''

### Dictionary to keep track of training progress

In [12]:
yaml_results = {}
yaml_results['training'] = {}
yaml_results['initialization'] = {}
yaml_results['plotting'] = {}
yaml_results['training']['status'] = 'failed' # Change this to 'Successful' at the end of the script
yaml_results['initialization']['status'] = 'failed' # Change this to 'Successful' after calculating initialization loss
yaml_results['plotting']['status'] = 'failed'

### Load from checkpoint (optional)

In [13]:
if not load_checkpoint:
    if log_file == '':
        log_file = os.path.join('calibration_scripts_logs', f'exp_{exp_id}.yaml')
    with open(log_file, 'w') as f:
        yaml.safe_dump(yaml_results, f, sort_keys=False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu") # Empirically, CPU seems to work faster for optimization
print(f'Device: {device}')

Device: cpu


### Load camera intrinsics and pre-processed grid points data

In [28]:
pi = torch.tensor(np.pi, dtype=datatype).to(device)
calibration_results_file = 'dotted_grid_pairwise_data.mat'
calibration_results_path = os.path.join(calibration_results_dir, calibration_results_file)
prism_initialization_path = os.path.join(calibration_results_dir, 'prism_initialization.mat')
prism_image_name = 'initialization'
if os.path.isfile(
    os.path.join(
    calibration_results_dir,
    'calibration_grid_images',
    f'{prism_image_name}_cam_0.bmp')
    ):
    prism_image_cam_0_path = os.path.join(
    calibration_results_dir,
    'calibration_grid_images',
    f'{prism_image_name}_cam_0.bmp',
    )
else:
    prism_image_cam_0_path = os.path.join(
    calibration_results_dir,
    'calibration_grid_images',
    f'{prism_image_name}_cam_0.png'
)
if os.path.isfile(
    os.path.join(
    calibration_results_dir,
    'calibration_grid_images',
    f'{prism_image_name}_cam_1.bmp')
    ):
    prism_image_cam_1_path = os.path.join(
    calibration_results_dir,
    'calibration_grid_images',
    f'{prism_image_name}_cam_1.bmp',
    )
else:
    prism_image_cam_1_path = os.path.join(
    calibration_results_dir,
    'calibration_grid_images',
    f'{prism_image_name}_cam_1.png',
)
prism_image_cam_0 = plt.imread(prism_image_cam_0_path)    
prism_image_cam_1 = plt.imread(prism_image_cam_1_path)
print('Loaded prism initialization data from {prism_initialization_path}')
outputs_dir = 'outputs'
os.makedirs(outputs_dir, exist_ok=True)
now = datetime.datetime.now()

if load_checkpoint:
    model_checkpoint_dir = input_model_checkpoint_dir 
    #model_checkpoint_dir = '/groups/branson/bransonlab/aniket/fly_walk_imaging/calibration_code/refraction_model/calprism/outputs/model_checkpoints/exp_62_2025_9_4_12_12_48'
else:
    model_checkpoint_dir = f'{outputs_dir}/model_checkpoints/exp_{exp_id}_{now.year}_{now.month}_{now.day}_{now.hour}_{now.minute}_{now.second}'
    os.makedirs(model_checkpoint_dir, exist_ok=True)
    yaml_results['model_checkpoint_dir'] = model_checkpoint_dir

num_points = -1
mat = sio.loadmat(calibration_results_path)
virtual_pixels_cam_0 = torch.tensor(mat['output_data_cam_02_pairwise'], dtype=datatype).T - 1.
virtual_pixels_cam_1 = torch.tensor(mat['output_data_cam_13_pairwise'], dtype=datatype).T - 1.
undistorted_real_pixels_cam_0 = torch.tensor(mat['output_data_cam_0_pairwise'], dtype=datatype).T - 1.
undistorted_real_pixels_cam_1 = torch.tensor(mat['output_data_cam_1_pairwise'], dtype=datatype).T - 1.
target_coordinates = torch.tensor(mat['worldPoints_pairwise'], dtype=datatype).T
pairwise_distance = torch.tensor(mat['pairwise_distances'][:,0]).to(datatype)
virtual_pixels_cam_0 = virtual_pixels_cam_0[:, :num_points]
virtual_pixels_cam_1 = virtual_pixels_cam_1[:, :num_points]
undistorted_real_pixels_cam_0 = undistorted_real_pixels_cam_0[:, :num_points]
undistorted_real_pixels_cam_1 = undistorted_real_pixels_cam_1[:, :num_points]
target_coordinates = target_coordinates[:, :num_points]
pairwise_distance = pairwise_distance[:num_points]

stereoParams = mat['stereoParams_export']
K1 = torch.tensor(stereoParams['CameraParameters1K'][0,0]).to(datatype).to(device)
K2 = torch.tensor(stereoParams['CameraParameters2K'][0,0]).to(datatype).to(device)
R = torch.tensor(stereoParams['RotationOfCamera2'][0,0]).to(datatype).to(device)
T = torch.tensor(stereoParams['TranslationOfCamera2'][0,0]).to(datatype).T.to(device)
radial_dist_coeffs_cam_0 = torch.tensor(stereoParams['RadialDistortionOfCamera1'][0,0]).to(datatype).T.to(device)
radial_dist_coeffs_cam_1 = torch.tensor(stereoParams['RadialDistortionOfCamera2'][0,0]).to(datatype).T.to(device)

principal_point_pixel_cam_0 = torch.tensor([K1[0,2] - 1, K1[1,2] - 1], dtype=datatype).to(device)
principal_point_pixel_cam_1 = torch.tensor([K2[0,2] - 1, K2[1,2] - 1], dtype=datatype).to(device)


focal_length_cam_1 = (K1[0,0] + K1[1,1]) /  2
focal_length_cam_2 = (K2[0,0] + K2[1,1]) /  2

# Tensorboard writer
writer = SummaryWriter(log_dir=f'{model_checkpoint_dir}')
print(f'To visualize tensorboard log, run: tensorboard --logdir={outputs_dir}/logs/{model_checkpoint_dir}')

def change_lr(optimizer, lr):
    for param_group in optimizer.param_groups:
            param_group['lr'] = lr

Loaded prism initialization data from {prism_initialization_path}
To visualize tensorboard log, run: tensorboard --logdir=outputs/logs/outputs/model_checkpoints/exp_exp_32_2025_12_2_12_56_58


### Freeze a subset of parameters that are already estimated apriori. These are fine-tuned after a few epochs

In [29]:
def freeze_camera_parameters(camera):
    for param in camera.parameters():
        param.requires_grad = False

def freeze_individual_planes(prism):
    for param in prism.plane1.parameters():
        param.requires_grad = False
    
    for param in prism.plane2.parameters():
        param.requires_grad = False
    
    for param in prism.plane3.parameters():
        param.requires_grad = False

def freeze_stereocamera(arena):
    arena.focal_length_cam_1.requires_grad = False
    arena.principal_point_pixel_cam_1.requires_grad = False
    arena.stereo_camera_rotation_6d.rotation_6d.requires_grad = False
    arena.stereocam_r1.requires_grad = False
    arena.T_stereo_cam.requires_grad = False

def freeze_prism_parameters_subset(arena):
    arena.prism.prism_size.requires_grad = False
    arena.prism.refractive_index_glass.requires_grad = False

def unfreeze_camera_parameters(camera):
    for param in camera.parameters():
        param.requires_grad = True

def unfreeze_stereocamera(arena):
    arena.focal_length_cam_1.requires_grad = True
    arena.principal_point_pixel_cam_1.requires_grad = True
    arena.stereo_camera_rotation_6d.requires_grad = True
    arena.stereocam_r1.requires_grad = True

def unfreeze_prism_parameters_subset(prism):
    for param in prism.parameters():
        param.requires_grad = True

def unfreeze_all_parameters(arena):
    for param in arena.parameters():
        if not param.requires_grad:
            param.requires_grad = True

### Initialize the prism-cameras arena using exported .mat data
##### This initialization is performed using camera intrinsics computed apriori, and a user-provided image of the grid sitting on the prism at a known location

In [34]:
prism_initializations = sio.loadmat(prism_initialization_path)
prism_center = torch.tensor(prism_initializations['location_prism']).to(datatype).T.to(device)
plane = Plane(axes=prism_axes)
prism_angles = torch.tensor([plane.alpha, plane.beta, plane.gamma], dtype=datatype).to(device)

arena = Arena_reprojection_loss_two_cameras_prism_grid_distances(principal_point_pixel_cam_0,
            principal_point_pixel_cam_1, 
            focal_length_cam_1, 
            focal_length_cam_2,
            R,
            T, 
            prism_angles=prism_angles,
            prism_center=prism_center,
            radial_dist_coeffs_cam_0=radial_dist_coeffs_cam_0,
            radial_dist_coeffs_cam_1=radial_dist_coeffs_cam_1,
            prism_size=torch.tensor([80., 20., 20.], dtype=datatype).to(device))
pixels_virtual_two_cams = torch.vstack((virtual_pixels_cam_0, virtual_pixels_cam_1))
pixels_real_two_cams = torch.vstack((undistorted_real_pixels_cam_0, undistorted_real_pixels_cam_1))
freeze_camera_parameters(arena.camera1)
freeze_stereocamera(arena)
freeze_prism_parameters_subset(arena)



### Visualize arena

In [ ]:
if load_checkpoint:
    PATH = f'{model_checkpoint_dir}/best_checkpoint.pth'
    checkpoint = torch.load(PATH, weights_only=True)
    arena.load_state_dict(checkpoint['model_state_dict'],
                          strict=False)

front_corner, hypotenuse_corner, top_corner = arena.get_prism_corners()
R1 = torch.eye(3, 3).to(device=pixels_virtual_two_cams.device, dtype=datatype).to(device)
T1 = torch.zeros(3, 1).to(device=pixels_virtual_two_cams.device, dtype=datatype).to(device)
front_corner_2D = arena.camera1.reproject(front_corner, R1, T1).to(device)
front_corner_2D = arena.camera1.distort_pixels_classical(
    front_corner_2D,
    arena.radial_dist_coeffs_cam_0,
)
hypotenuse_corner_2D = arena.camera1.reproject(hypotenuse_corner, R1, T1).to(device)
hypotenuse_corner_2D = arena.camera1.distort_pixels_classical(
    hypotenuse_corner_2D,
    arena.radial_dist_coeffs_cam_0,
)

top_corner_2D = arena.camera1.reproject(top_corner, R1, T1).to(device)
top_corner_2D = arena.camera1.distort_pixels_classical(
    top_corner_2D,
    arena.radial_dist_coeffs_cam_0,
)
plt.imshow(prism_image_cam_0, cmap='gray')
plt.gca().axis('off')
plt.plot(
    front_corner_2D[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    front_corner_2D[1][[0,1,2,3,0]].cpu().detach().numpy(),
    linewidth=2,
)
plt.plot(
    hypotenuse_corner_2D[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    hypotenuse_corner_2D[1][[0,1,2,3,0]].cpu().detach().numpy(),
    linewidth=2,
)
plt.plot(
    top_corner_2D[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    top_corner_2D[1][[0,1,2,3,0]].cpu().detach().numpy(),
    linewidth=2,
)

R_stereo_cam = arena.stereo_camera_rotation_6d.matrix()
R2 = R_stereo_cam
T2 = arena.T_stereo_cam
camera2 = arena.get_stereo_camera(arena.principal_point_pixel_cam_1,
                            arena.focal_length_cam_1,
                            R2,
                            arena.T_stereo_cam,
                            r1=arena.stereocam_r1,
                            radial_dist_coeffs=arena.radial_dist_coeffs_cam_1)
front_corner_2D = camera2.reproject(front_corner, R2, T2).to(device)
front_corner_2D = camera2.distort_pixels_classical(
    front_corner_2D,
    arena.radial_dist_coeffs_cam_1,
)
hypotenuse_corner_2D = camera2.reproject(hypotenuse_corner, R2, T2).to(device)
hypotenuse_corner_2D = camera2.distort_pixels_classical(
    hypotenuse_corner_2D,
    arena.radial_dist_coeffs_cam_1,
)
top_corner_2D = camera2.reproject(top_corner, R2, T2).to(device)
top_corner_2D = camera2.distort_pixels_classical(
    top_corner_2D,
    arena.radial_dist_coeffs_cam_1,
)
plt.figure()
plt.imshow(prism_image_cam_1, cmap='gray')
plt.plot(
    front_corner_2D[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    front_corner_2D[1][[0,1,2,3,0]].cpu().detach().numpy(),
    linewidth=2,
)
plt.plot(
    hypotenuse_corner_2D[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    hypotenuse_corner_2D[1][[0,1,2,3,0]].cpu().detach().numpy(),
    linewidth=2,
)
plt.plot(
    top_corner_2D[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    top_corner_2D[1][[0,1,2,3,0]].cpu().detach().numpy(),
    linewidth=2,
)

### Training and validation functions

In [31]:
def train_two_cams(model, train_loader, criterion, plot=False):    
    model.train()
    virtual_loss = 0.
    real_loss = 0.
    closest_dist_loss = 0.
    total_loss = 0.
    distortion_loss = 0.
    intersection_loss = 0.
    reprojection_loss = 0.
    tr_loss = 0.
    pairwise_dist_loss = 0.
    calibration_skew_loss = 0.
    
    with torch.autograd.set_detect_anomaly(True):
        # Iterate over minibatches
        if plot:
            plt.figure()
        for i, (input, label_2D, label_3D, pairwise_distance_batch) in enumerate(train_loader):
            num_examples = input.shape[0]       
            optimizer.zero_grad()
            input = input.to(device)
            label_2D = label_2D.to(device)
            label_3D = label_3D.to(device)
            pairwise_distance_batch = pairwise_distance_batch.to(device)
            output = model(
                input.T,
                label_2D.T)
            recon_3D, closest_distance, recon_pixels_1, recon_pixels_2, recon_pixels_1_from_virtual, recon_pixels_2_from_virtual, recon_pixels_1_to_virtual, recon_pixels_2_to_virtual, dist_penalty_1, dist_penalty_2, int_penalty_1, int_penalty_2, pairwise_distance_recon = output['recon_3D'], output['closest_distance'], output['recon_pixels_1'], output['recon_pixels_2'], output['recon_pixels_0_virtual'], output['recon_pixels_1_virtual'], output['recon_pixels_1_to_virtual'], output['recon_pixels_2_to_virtual'], output['distortion_penalty_cam_0'], output['distortion_penalty_cam_1'], output['intersection_penalty_1'], output['intersection_penalty_2'], output['pairwise_distance']
            d_pairwise_distance = (pairwise_distance_recon - pairwise_distance_batch)
            pairwise_distance_loss = torch.abs(d_pairwise_distance).sum() # Sum of root squared error
            label_2D_cam_0 = torch.vstack((label_2D[:,:2], label_2D[:,2:4]))
            label_2D_cam_1 = torch.vstack((label_2D[:,4:6], label_2D[:,6:8]))
            virtual_2D_cam_0 = torch.vstack((input[:,:2], input[:,2:4]))
            virtual_2D_cam_1 = torch.vstack((input[:,4:6], input[:,6:8]))
            stacked_label_2D = torch.vstack((label_2D[:,:label_2D.shape[1] // 2], label_2D[:,label_2D.shape[1] // 2:]))
            stacked_label_3D = torch.vstack((label_3D[:,:label_3D.shape[1] // 2], label_3D[:,label_3D.shape[1] // 2:]))
            if plot:                
                rand_ind = torch.randperm(recon_pixels_1.shape[1])
                plt.subplot(121)
                plt.scatter(
                    recon_pixels_1[0,rand_ind].cpu().detach().numpy(),
                    recon_pixels_1[1,rand_ind].cpu().detach().numpy(),
                    s=0.5,
                    c='r',
                )
                plt.scatter(
                    label_2D_cam_0.T[0,rand_ind].cpu().detach().numpy(),
                    label_2D_cam_0.T[1,rand_ind].cpu().detach().numpy(),
                    s=0.5,
                    c='g',
                )
                plt.subplot(122)
                plt.scatter(
                    recon_pixels_2[0,rand_ind].cpu().detach().numpy(),
                    recon_pixels_2[1,rand_ind].cpu().detach().numpy(),
                    s=0.5,
                    c='r',
                )
                plt.scatter(
                    label_2D_cam_1.T[0,rand_ind].cpu().detach().numpy(),
                    label_2D_cam_1.T[1,rand_ind].cpu().detach().numpy(),
                    s=0.5,
                    c='g',
                )


            if recon_pixels_1_to_virtual is not None and recon_pixels_2_to_virtual is not None:
                print('Calculated virtual reprojection error')
                
                overall_reprojection_loss = (euclidean_distance(
                        recon_pixels_1_to_virtual, 
                        virtual_2D_cam_0.T).sum() + euclidean_distance(
                        recon_pixels_2_to_virtual, 
                        virtual_2D_cam_1.T).sum()
                    ) / 4
                
            else:
                if torch.rand(1) < .5:
                    overall_reprojection_loss = (euclidean_distance(
                        recon_pixels_1, 
                        label_2D_cam_0.T).sum() + euclidean_distance(
                        recon_pixels_2,
                        label_2D_cam_1.T).sum()) / 4
                else:
                    overall_reprojection_loss = (euclidean_distance(
                        recon_pixels_1_from_virtual, 
                        label_2D_cam_0.T).sum() + euclidean_distance(
                        recon_pixels_2_from_virtual,
                        label_2D_cam_1.T).sum()) / 4
            

            triangulation_loss = euclidean_distance(
                                    recon_3D, stacked_label_3D.T
                                ).sum() / 2
            
            distortion_loss = (dist_penalty_1.sum() + dist_penalty_2.sum()) / 2
            intersection_loss = (int_penalty_1.sum() + int_penalty_2.sum()) / 2
            closest_distance_loss = closest_distance.sum()
            tr_loss += triangulation_loss.item()            
            # Recon_real_loss is the pixel error between the reprojected 3D point from real pixels and the real pixel
            loss = (1e1 * overall_reprojection_loss) + (1e10 * intersection_loss) + 0 * distortion_loss + (1e2 * closest_distance_loss) + (1e4 * pairwise_distance_loss) 
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            # virtual_loss += recon_virtual_loss.item()
            # real_loss += recon_real_loss.item()
            closest_dist_loss += closest_distance_loss.item()
            distortion_loss += distortion_loss.item()
            intersection_loss += intersection_loss.item()
            reprojection_loss += overall_reprojection_loss.item()     
            pairwise_dist_loss += pairwise_distance_loss.item()      

        virtual_loss = 0.
        real_loss = 0.

    return total_loss / len(train_loader.dataset), reprojection_loss / len(train_loader.dataset), virtual_loss / len(train_loader.dataset), real_loss / len(train_loader.dataset), closest_dist_loss / len(train_loader.dataset), distortion_loss / len(train_loader.dataset), intersection_loss / len(train_loader.dataset), tr_loss / len(train_loader.dataset), pairwise_dist_loss / len(train_loader.dataset)


def validate(model, val_loader, criterion):
    model.eval()
    virtual_loss = 0.
    real_loss = 0.
    closest_dist_loss = 0.
    total_loss = 0.
    distortion_loss = 0.
    intersection_loss = 0.
    tr_loss = 0.
    reprojection_loss = 0.
    pairwise_dist_loss = 0.
    with torch.no_grad():
        for i, (input, label_2D, label_3D, pairwise_distance_batch) in enumerate(val_loader):
            # input: virtual pixels from two cameras stacked
            # label_2D: real pixels from two cameras stacked
            # label_3D: target 3D coordinates stacked
            # pairwise_distance_batch: pairwise distances between points in the batch
            input = input.to(device)
            label_2D = label_2D.to(device)
            label_3D = label_3D.to(device)
            pairwise_distance_batch = pairwise_distance_batch.to(device)
            output = model(input.T, label_2D.T)
            recon_3D, closest_distance, recon_pixels_1, recon_pixels_2, recon_pixels_1_to_virtual, recon_pixels_2_to_virtual, dist_penalty_1, dist_penalty_2, int_penalty_1, int_penalty_2, pairwise_distance_recon = output['recon_3D'], output['closest_distance'], output['recon_pixels_1'], output['recon_pixels_2'], output['recon_pixels_1_to_virtual'], output['recon_pixels_2_to_virtual'], output['distortion_penalty_cam_0'], output['distortion_penalty_cam_1'], output['intersection_penalty_1'], output['intersection_penalty_2'], output['pairwise_distance']
            d_pairwise_distance = (pairwise_distance_recon - pairwise_distance_batch)
            pairwise_distance_loss = torch.abs(d_pairwise_distance).sum() # Sum of root squared erro

            label_2D_cam_0 = torch.vstack((label_2D[:,:2], label_2D[:,2:4]))
            label_2D_cam_1 = torch.vstack((label_2D[:,4:6], label_2D[:,6:8]))
            virtual_2D_cam_0 = torch.vstack((input[:,:2], input[:,2:4]))
            virtual_2D_cam_1 = torch.vstack((input[:,4:6], input[:,6:8]))
            stacked_label_3D = torch.vstack((label_3D[:,:label_3D.shape[1] // 2], label_3D[:,label_3D.shape[1] // 2:]))

            
            if recon_pixels_1_to_virtual is not None and recon_pixels_2_to_virtual is not None:
                print('Calculated virtual reprojection error')
                overall_reprojection_loss = (euclidean_distance(
                        recon_pixels_1_to_virtual, 
                        virtual_2D_cam_0.T).sum() + euclidean_distance(
                        recon_pixels_2_to_virtual, 
                        virtual_2D_cam_1.T).sum() + euclidean_distance(
                        recon_pixels_1,
                        label_2D_cam_0.T).sum() + euclidean_distance(
                        recon_pixels_2,
                        label_2D_cam_1.T).sum()
                    ) / 8

            else:
                overall_reprojection_loss = (euclidean_distance(
                recon_pixels_1, 
                label_2D_cam_0.T).sum() + euclidean_distance(
                recon_pixels_2,
                label_2D_cam_1.T).sum()) / 4
                   
            triangulation_loss = euclidean_distance(
                                    recon_3D, stacked_label_3D.T
                                ).sum() / 2
            
            distortion_loss = (dist_penalty_1.sum() + dist_penalty_2.sum()) / 2
            intersection_loss = (int_penalty_1.sum() + int_penalty_2.sum()) / 2
            closest_distance_loss = closest_distance.sum()
            loss = (1e1 * overall_reprojection_loss) + (1e10 * intersection_loss) + (0 * distortion_loss) + (1e1 * closest_distance_loss) + (5e3 * pairwise_distance_loss)

            total_loss += loss.item()
            #virtual_loss += recon_virtual_loss.item()
            #real_loss += recon_real_loss.item()
            intersection_loss += intersection_loss.item()
            distortion_loss += distortion_loss.item()
            closest_dist_loss += closest_distance_loss.item()
            tr_loss += triangulation_loss.item()
            reprojection_loss += overall_reprojection_loss.item()
            pairwise_dist_loss += pairwise_distance_loss.item()

        virtual_loss = 0.
        real_loss = 0.

    return total_loss / len(val_loader.dataset), reprojection_loss / len(val_loader.dataset), virtual_loss / len(val_loader.dataset), real_loss / len(val_loader.dataset), closest_dist_loss / len(val_loader.dataset), distortion_loss / len(val_loader.dataset), intersection_loss / len(val_loader.dataset), tr_loss / len(val_loader.dataset), pairwise_dist_loss / len(val_loader.dataset)

### Visualize the initialized arena with a randomly sampled subset of inverse-traced rays

In [32]:
arena.visualize(pixels_virtual_two_cams.to(device), color_labels=True)
#plt.savefig(f'{outputs_dir}/initialized_arena.png')
output = arena(
    pixels_virtual_two_cams.to(device),
    pixels_real_two_cams.to(device)
)
recon_3D, closest_distance, recon_pixels_1, recon_pixels_2, pairwise_distance_ = output['recon_3D'], output['closest_distance'], output['recon_pixels_1'], output['recon_pixels_2'], output['pairwise_distance']

pairwise_distance_loss = torch.abs(pairwise_distance_ - pairwise_distance).mean()
pixels_real_cam_0_test_stacked = torch.hstack((pixels_real_two_cams[:2,:], pixels_real_two_cams[2:4,:]))
pixels_real_cam_1_test_stacked = torch.hstack((pixels_real_two_cams[4:6,:], pixels_real_two_cams[6:8,:]))
recon_real_loss = (euclidean_distance(
                recon_pixels_1, 
                pixels_real_cam_0_test_stacked).mean() + euclidean_distance(
                recon_pixels_2, 
                pixels_real_cam_1_test_stacked).mean()) / 2
target_coordinates_stacked = torch.hstack((target_coordinates[:3,:], target_coordinates[3:,:]))
triangulation_loss = euclidean_distance(
                        recon_3D, target_coordinates_stacked
                    ).mean()    
print(f'Initial real pixel reprojection error: {recon_real_loss}, initial closest distance: {closest_distance.mean()}, initial pairwise distance error: {pairwise_distance_loss}, initial triangulation error: {triangulation_loss}')
yaml_results['initialization']['status'] = 'passed'
yaml_results['initialization']['repr_error_real'] = recon_real_loss.item()
yaml_results['initialization']['closest_distance_error'] = closest_distance.mean().item()
yaml_results['initialization']['pairwise_distance_error'] = pairwise_distance_loss.item()
if not load_checkpoint:
    with open(log_file, 'w') as f:
        yaml.safe_dump(yaml_results, f, sort_keys=False)

/groups/branson/bransonlab/aniket/fly_walk_imaging/calibration_code/refraction_model/calprism/ray_tracing_simulator_nnModules_grad.py:600: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(sampled_points[0],


Initial real pixel reprojection error: 31.45532467266181, initial closest distance: 0.31568657493478913, initial pairwise distance error: 0.024528981048087862, initial triangulation error: 1.5945076856085703


### Load dataset

In [ ]:
batch_size=1024
batch_size=2048
pixels_virtual_two_cams = pixels_virtual_two_cams.to(device)
pixels_real_two_cams = pixels_real_two_cams.to(device)
target_coordinates = target_coordinates.to(device)
pairwise_distance = pairwise_distance.to(device)
rand_ind = torch.randperm(pixels_virtual_two_cams.shape[1]).to(device)
test_dataset_size = 150
pixels_virtual_two_cams_test = pixels_virtual_two_cams[:, rand_ind[:test_dataset_size]]
target_coordinates_test = target_coordinates[:, rand_ind[:test_dataset_size]]
pixels_real_two_cams_test = pixels_real_two_cams[:, rand_ind[:test_dataset_size]]
pairwise_distance_test = pairwise_distance[rand_ind[:test_dataset_size]]

pixels_virtual_two_cams = pixels_virtual_two_cams[:,rand_ind[test_dataset_size:]]
pixels_real_two_cams = pixels_real_two_cams[:, rand_ind[test_dataset_size:]]
target_coordinates = target_coordinates[:, rand_ind[test_dataset_size:]]
pairwise_distance_train_val = pairwise_distance[rand_ind[test_dataset_size:]]

dataset = CalibrationDataset(pixels_virtual_two_cams, pixels_real_two_cams, target_coordinates, pairwise_distance_train_val)
train_size = int(0.8 * len(dataset))  # 80% for training
val_size = len(dataset) - train_size   # Remaining 20% for validation
print(f'Splitting grid point pairs into {train_size} : {val_size} ratio')
pixels_virtual_two_cams_train, pixels_virtual_two_cams_val = random_split(
    dataset, [train_size, val_size]
    )
train_loader = DataLoader(pixels_virtual_two_cams_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(pixels_virtual_two_cams_val, batch_size=batch_size, shuffle=False)

optimizer = optim.Adam(arena.parameters(), lr=1e-2
                       )
criterion = torch.nn.MSELoss()
arena.to(device)
pixels_virtual_two_cams = pixels_virtual_two_cams.to(device)

### Training loop

In [ ]:
train_loss_array = []
train_virtual_loss_array = []
train_real_loss_array = []
train_closest_distance_loss_array = []
train_distortion_loss_array = []
train_intersection_loss_array = []

val_loss_array = []
val_virtual_loss_array = []
val_real_loss_array = []
val_closest_distance_loss_array = []
val_distortion_loss_array = []
val_intersection_loss_array = []

gt_train_loss_array = []
closest_distance_train_loss_array = []
gt_val_loss_array = []
closest_distance_val_loss_array = []
best_loss = 1e100

plot = False
arena.virtual_proj_prob_thresh = 0. # probability of calculating virtual reprojection error and using it for backprop
for epoch in tqdm(range(0, num_epochs)):
    if epoch == 75:
        change_lr(optimizer, lr=1e-2)

    if epoch == 150:
        change_lr(optimizer, lr=5e-3)

    if epoch == 200:
        change_lr(optimizer, lr=1e-3)

    if epoch == 250:
        change_lr(optimizer, lr=5e-4)

    if epoch == 300:
        change_lr(optimizer, lr=1e-4)
        unfreeze_all_parameters(arena)
    
    if epoch == 800:
        change_lr(optimizer, lr=5e-5)

    train_loss, train_reprojection_loss, train_virtual_loss, train_real_loss, train_closest_dist_loss, train_distortion_loss, train_intersection_loss, triangulation_loss, train_pairwise_distance_loss = train_two_cams(model=arena, 
                    train_loader=train_loader, 
                    criterion=criterion,
                    plot=plot
                    )
    
    if plot:
        plt.title(f'Epoch {epoch}')
    plot = False
    if epoch % 10 == 0:
        print(f'Training loss for epoch {epoch}: train_loss: {train_loss}, reprojection loss: {train_reprojection_loss}, closest_dist_loss: {train_closest_dist_loss}, triangulation loss : {triangulation_loss}, intersection loss: {train_intersection_loss}, distortion loss: {train_distortion_loss}')
        if epoch % 200 == 0:
            plot = True
    train_loss_array.append(train_loss)
    train_virtual_loss_array.append(train_virtual_loss)
    train_real_loss_array.append(train_real_loss)
    train_closest_distance_loss_array.append(train_closest_dist_loss)
    train_distortion_loss_array.append(train_distortion_loss)
    train_intersection_loss_array.append(train_intersection_loss)


    val_loss, val_reprojection_loss, val_virtual_loss, val_real_loss, val_closest_dist_loss, val_distortion_loss, val_intersection_loss, triangulation_loss, val_pairwise_distance_loss = validate(model=arena,
             val_loader=val_loader,
             criterion=criterion,
             )
    
    writer.add_scalar('Loss/val', val_loss, epoch)
    writer.add_scalar('Loss/val_reprojection_error', val_reprojection_loss, epoch)
    writer.add_scalar('Loss/val_closest_distance_error', val_distortion_loss, epoch)
    writer.add_scalar('Loss/val_triangulation_error', triangulation_loss, epoch)
    writer.add_scalar('Loss/val_pairwise_distance_error', val_pairwise_distance_loss, epoch)
    writer.add_scalar('Loss/val_intersection_error', val_intersection_loss, epoch)
    writer.add_scalar('Loss/val_distortion_error', val_distortion_loss, epoch)
    writer.add_scalar('Parameter/prism/refractive_index_glass', arena.prism.refractive_index_glass, epoch)

    prism_alpha, prism_beta, prism_gamma = arena.prism_rotation_6d.to_euler()
    writer.add_scalars('Parameter/prism_angles', {
        'Angle0':prism_alpha,
         'Angle1':prism_beta,
          'Angle2':prism_gamma},
            epoch)
    writer.add_scalars('Parameter/prism_size', {
        'Size0':arena.prism.prism_size[0],
         'Size1':arena.prism.prism_size[1],
          'Size2':arena.prism.prism_size[2]},
            epoch)
    writer.add_scalars('Parameter/prism_center', {
        'Center0':arena.prism.prism_center[0],
         'Center1':arena.prism.prism_center[1],
          'Center2':arena.prism.prism_center[2]},
            epoch)
    writer.add_scalar('Parameter/focal_length_pixels_0', arena.camera1.focal_length_pixels, epoch)
    writer.add_scalars('Parameter/camera1_principal_point', {
        'Angle0':arena.camera1.principal_point_pixel[0],
         'Angle1':arena.camera1.principal_point_pixel[1]},
            epoch)

    if epoch % 10 == 0:
        print(f'Validation loss for epoch {epoch}: val_loss: {val_loss}, reprojection error: {val_reprojection_loss}, closest_dist_loss: {val_closest_dist_loss}, triangulation loss {triangulation_loss}, distortion loss: {val_distortion_loss}, intersection loss: {val_intersection_loss}')
        # Save checkpoint
        torch.save({
                    'epoch': epoch,  # Save the current epoch
                    'model_state_dict': arena.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': train_loss,
                    'val_loss': val_loss,
                    'reprojection_loss': val_reprojection_loss,
                    'closest_dist_loss': val_closest_dist_loss,
                    'distortion_loss': val_distortion_loss,
                    'intersection_loss': val_intersection_loss,
                    'train_pairwise_distance_loss': train_pairwise_distance_loss,
                }, f'{model_checkpoint_dir}/checkpoint_{epoch}.pth')
                
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({
                    'epoch': epoch,  # Save the current epoch
                    'model_state_dict': arena.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': val_loss,
                }, f'{model_checkpoint_dir}/best_checkpoint.pth'),
        torch.save(
            arena.state_dict(),
            f'{model_checkpoint_dir}/best_model_weights_only.pth',
            _use_new_zipfile_serialization=False,
        )
        print(f'Found better model with validation loss for epoch {epoch}: val_loss: {val_loss}, reprojection error: {val_reprojection_loss}, closest_dist_loss: {val_closest_dist_loss}, triangulation loss {triangulation_loss}, pairwise_distance_loss: {val_pairwise_distance_loss}, intersection loss: {val_intersection_loss}')
    gt_train_loss_array.append(train_loss)
    gt_val_loss_array.append(val_loss)
    closest_distance_train_loss_array.append(train_closest_dist_loss)
    closest_distance_val_loss_array.append(val_closest_dist_loss)

### Evaluation of the trained model

In [ ]:

PATH = f'{model_checkpoint_dir}/best_checkpoint.pth'
checkpoint = torch.load(PATH, weights_only=True)
arena.load_state_dict(checkpoint['model_state_dict'],strict=False)

arena.virtual_proj_prob_thresh = 1.
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

output = arena(pixels_virtual_two_cams_test, pixels_real_two_cams_test)
recon_3D_test, closest_dist_test, recon_pixels_1, recon_pixels_2, recon_pixels_1_to_virtual, recon_pixels_2_to_virtual, recon_3D_real, real_3D_virtual, dist_penalty_1, dist_penalty_2, int_penalty_1, int_penalty_2, pairwise_distance_test_ = output['recon_3D'], output['closest_distance'], output['recon_pixels_1'], output['recon_pixels_2'], output['recon_pixels_1_to_virtual'], output['recon_pixels_2_to_virtual'], output['recon_3D_real'], output['recon_3D_virtual'], output['distortion_penalty_cam_0'], output['distortion_penalty_cam_1'], output['intersection_penalty_1'], output['intersection_penalty_2'], output['pairwise_distance']
pairwise_distance_loss = torch.abs(pairwise_distance_test_ - pairwise_distance_test)
target_coordinates_test_stacked = torch.hstack((target_coordinates_test[:3,:], target_coordinates_test[3:,:]))
triangulation_loss = euclidean_distance(
    recon_3D_test, target_coordinates_test_stacked
).mean()

pixels_real_cam_0_test_stacked = torch.hstack((pixels_real_two_cams_test[:2,:], pixels_real_two_cams_test[2:4,:]))
pixels_real_cam_1_test_stacked = torch.hstack((pixels_real_two_cams_test[4:6,:], pixels_real_two_cams_test[6:8,:]))
recon_real_loss = (euclidean_distance(
                recon_pixels_1, 
                pixels_real_cam_0_test_stacked).mean() + euclidean_distance(
                recon_pixels_2, 
                pixels_real_cam_1_test_stacked).mean()) / 2

print(f'Triangulation loss: {triangulation_loss}')
print(f'Distortion penalty: {(dist_penalty_1 + dist_penalty_2).mean()}')
print(f'Real pixel loss: {recon_real_loss.mean()}')
print(f'Pairwise distance loss: {pairwise_distance_loss.mean()}')
print(f'Intersection penalty: {(int_penalty_1 + int_penalty_2).mean() / 2}')

reprojection_loss_1 = euclidean_distance(
    recon_pixels_1, pixels_real_cam_0_test_stacked
)
reprojection_loss_2 = euclidean_distance(
    recon_pixels_2, pixels_real_cam_1_test_stacked
)
print(f'Reprojection error for two cameras: {reprojection_loss_1.mean()}, {reprojection_loss_2.mean()}')

if (recon_pixels_1_to_virtual is not None) and (recon_pixels_2_to_virtual is not None):
    print('Calculated virtual reprojection error')
    pixels_virtual_cam_0_test_stacked = torch.hstack((pixels_virtual_two_cams_test[:2,:], pixels_virtual_two_cams_test[2:4,:]))
    pixels_virtual_cam_1_test_stacked = torch.hstack((pixels_virtual_two_cams_test[4:6,:], pixels_virtual_two_cams_test[6:8,:]))
    reprojection_loss_1_to_virtual = euclidean_distance(
        recon_pixels_1_to_virtual, pixels_virtual_cam_0_test_stacked
    )
    reprojection_loss_2_to_virtual = euclidean_distance(
        recon_pixels_2_to_virtual, pixels_virtual_cam_1_test_stacked
    )
    print(f'Reprojection error for two virtual cameras: {reprojection_loss_1_to_virtual.mean()}, {reprojection_loss_2_to_virtual.mean()}')

if not load_checkpoint:
    yaml_results['training']['status'] = 'passed'
    yaml_results['training']['repr_error_real_cam_0'] = reprojection_loss_1.mean().item()
    yaml_results['training']['repr_error_real_cam_1'] = reprojection_loss_2.mean().item()
    yaml_results['training']['repr_error_virtual_cam_0'] = reprojection_loss_1_to_virtual.mean().item()
    yaml_results['training']['repr_error_virtual_cam_1'] = reprojection_loss_2_to_virtual.mean().item()
    yaml_results['training']['pairwise_distance_error'] = pairwise_distance_loss.mean().item()
    yaml_results['training']['triangulation_error'] = triangulation_loss.item()

    with open(log_file, 'w') as f:
        yaml.safe_dump(yaml_results, f, sort_keys=False)

plt.figure(figsize=(15,15))
plt.scatter(
    recon_pixels_1[0,:].cpu().detach().numpy(),
    recon_pixels_1[1,:].cpu().detach().numpy(),
    color='r',
    marker='o',
    label='Estimate',
)
plt.scatter(
    pixels_real_cam_0_test_stacked[0,:].cpu().detach().numpy(),
    pixels_real_cam_0_test_stacked[1,:].cpu().detach().numpy(),
    color='b',
    marker='x',
    label='Ground truth',
)
ax = plt.gca()
ax.set_aspect('equal')
ax.set_xlabel('X (pixels)', fontsize=22)
ax.set_ylabel('Y (pixels)', fontsize=22)
ax.set_xticklabels(ax.get_xticks(), fontsize=18)
ax.set_yticklabels(ax.get_yticks(), fontsize=18)
ax.set_title(f'Reprojection error: {reprojection_loss_1.mean():.2f} pixels', fontsize=16)
plt.legend(fontsize=18)
plt.savefig(f'{model_checkpoint_dir}/reprojection_loss_1.png')

plt.figure(figsize=(15,15))
plt.scatter(
    recon_pixels_2[0,:].cpu().detach().numpy(),
    recon_pixels_2[1,:].cpu().detach().numpy(),
    color='r',
    marker='o',
    label='Estimate',
)
plt.scatter(
    pixels_real_cam_1_test_stacked[0,:].cpu().detach().numpy(),
    pixels_real_cam_1_test_stacked[1,:].cpu().detach().numpy(),
    color='b',
    marker='x',
    label='Ground truth',
)
ax = plt.gca()
ax.set_aspect('equal')
ax.set_xlabel('X (pixels)', fontsize=22)
ax.set_ylabel('Y (pixels)', fontsize=22)
ax.set_xticklabels(ax.get_xticks(), fontsize=18)
ax.set_yticklabels(ax.get_yticks(), fontsize=18)
ax.set_title(f'Reprojection error: {reprojection_loss_2.mean():.2f} pixels', fontsize=16)
plt.legend(fontsize=18)
plt.savefig(f'{model_checkpoint_dir}/reprojection_loss_2.png')

### Visualize results

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
ax.scatter(
    target_coordinates_test_stacked[0,:].cpu().detach().numpy(),
    target_coordinates_test_stacked[1,:].cpu().detach().numpy(),
    target_coordinates_test_stacked[2,:].cpu().detach().numpy(),
    color='r',
    s=5,
    label='Ground truth',
)
ax.scatter(
    recon_3D_test[0,:].cpu().detach().numpy(),
    recon_3D_test[1,:].cpu().detach().numpy(),
    recon_3D_test[2,:].cpu().detach().numpy(),
    color='b',
    s=5,
    label='Estimate',
)
front_corner, hypotenuse_corner, top_corner = arena.get_prism_corners()
ax.plot(
    front_corner[0][[0,1,2,3,0]].cpu().detach().numpy(), 
    front_corner[1][[0,1,2,3,0]].cpu().detach().numpy(),
    front_corner[2][[0,1,2,3,0]].cpu().detach().numpy(),
)
ax.plot(
    hypotenuse_corner[0][[0,1,2,3,0]].cpu().detach().numpy(),
    hypotenuse_corner[1][[0,1,2,3,0]].cpu().detach().numpy(),
    hypotenuse_corner[2][[0,1,2,3,0]].cpu().detach().numpy(),
)
ax.plot(
    top_corner[0][[0,1,2,3,0]].cpu().detach().numpy(),
    top_corner[1][[0,1,2,3,0]].cpu().detach().numpy(),
    top_corner[2][[0,1,2,3,0]].cpu().detach().numpy(),
)
ax.set_xlabel('X (mm)', fontsize=22)
ax.set_ylabel('Y (mm)', fontsize=22)
ax.set_zlabel('Z (mm)', fontsize=22)
ax.set_xticklabels(ax.get_xticks(), fontsize=18)
ax.set_yticklabels(ax.get_yticks(), fontsize=18)
ax.set_zticklabels(ax.get_zticks(), fontsize=18)
plt.savefig(f'{model_checkpoint_dir}/3D_scatter.png')

In [ ]:
plt.figure(figsize=(15,15))
epochs = np.arange(0,len(gt_train_loss_array))
plt.plot(epochs,gt_train_loss_array, color='b', 
         label='Ground truth train loss')
plt.plot(epochs,closest_distance_train_loss_array, 
         color='r', 
         label='Closest distance train loss')
plt.plot(epochs, gt_val_loss_array, color='b', 
         label='Ground truth val loss',
         linestyle='--')
plt.plot(epochs, closest_distance_val_loss_array, 
         color='r', 
         label='Closest distance val loss',
         linestyle='--')
plt.legend(fontsize=18)
plt.xlabel('Epochs', fontsize=22)
plt.ylabel('Loss (mm)', fontsize=22)
ax.set_xticklabels(ax.get_xticks(), fontsize=18)
ax.set_yticklabels(ax.get_yticks(), fontsize=18)
ax.set_zticklabels(ax.get_zticks(), fontsize=18)
plt.savefig(f'{model_checkpoint_dir}/training_loss.png')

In [ ]:
arena.visualize(pixels_virtual_two_cams_test, color_labels=True)
plt.savefig(f'{model_checkpoint_dir}/final_arena.png')
yaml_results['plotting']['status'] = "passed"
with open(log_file, 'w') as f:
    yaml.safe_dump(yaml_results, f, sort_keys=False)

with open('temp.yaml', 'r') as file:
    data = yaml.load(file, Loader=yaml.FullLoader)
    
data['model_path'] = f'{model_checkpoint_dir}/best_model_weights_only.pth'

with open('temp.yaml', 'w') as file:
    yaml.dump(data, file)